# 01 - Dermatology CBM + Fairness-Weighted PPO Triage (walkthrough)

This notebook mirrors `01_train_cbm.py` cell-by-cell. It trains a frozen-backbone `DermatologyCBM` (HRNet-W18) on a stratified HAM10000 binary split (melanoma vs benign), embeds synthetic cases through the CBM, trains the `PPOAgent` triage policy, then runs a Fitzpatrick-grouped fairness audit.

Run it with a kernel rooted at the project folder (D:/Dermatology_AI). The dataset and metadata files load from absolute `D:/...` paths, so the working directory only decides which root is picked if `datasets` is not next to it.

## 0) Setup

Load the project modules, define the shared constants and paths, and create the checkpoint folder. Imports mirror `01_train_cbm.py` so the exact APIs confirmed earlier are reused: `DermDataset` from `datasetEquity`, `NUM_CONCEPTS` from `model`, `MelaninNormalization` + `dark_skin_aug` from `samplesEquity`, and the PPO pieces from `rl_module`.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if not (ROOT / 'datasets').exists():
    ROOT = Path('D:/Dermatology_AI')
sys.path.insert(0, str(ROOT))

from datasetEquity import DermDataset
from model import NUM_CONCEPTS
from samplesEquity import MelaninNormalization, dark_skin_aug
from rl_module import Case, DermTriageEnv, PPOAgent, Rollout, TRUE_LABEL_TO_TRIAGE

DATA_DIR = ROOT / 'datasets'
HAM_DIR = DATA_DIR / 'ham10000'
HAM_META = HAM_DIR / 'HAM10000_metadata.csv'
SYN_DIR = DATA_DIR / 'synthetic' / 'images'
SYN_META = DATA_DIR / 'synthetic' / 'synthetic_metadata.csv'
TRAIN_META = DATA_DIR / 'train_meta.csv'
VAL_META = DATA_DIR / 'val_meta.csv'
CKPT_DIR = ROOT / 'model_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CBM_CKPT = CKPT_DIR / 'cbm_binary.pt'
PPO_CKPT = CKPT_DIR / 'ppo_triage.pt'

IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
FITZ_PLACEHOLDER_HAM = 3

DEVICE = torch.device('cpu')
TORCH_SEED = 0
EPOCHS = 3
BATCH_SIZE = 32
EPISODES = 2000
UPDATE_INTERVAL = 64

random.seed(TORCH_SEED)
np.random.seed(TORCH_SEED)
torch.manual_seed(TORCH_SEED)
print('project root:', ROOT)
print('checkpoints ->', CKPT_DIR)

## 1) Stratified HAM10000 split

Binary label mapping: `dx == 'mel'` maps to `melanoma`, everything else maps to `benign`. We stratify on the original `dx` group, then take an 85/15 split inside each group so the melanoma rate is preserved in both splits. HAM10000 carries no Fitzpatrick skin type, so we record a neutral placeholder of 3 (type III).

Both CSVs use exactly the columns `DermDataset` expects: `image_id, label, fitzpatrick_type`. No sklearn dependency - a manual groupby split.

In [ ]:
def make_split(test_size=0.15, seed=TORCH_SEED):
    meta = pd.read_csv(HAM_META)
    meta['label'] = meta['dx'].map(lambda d: 'melanoma' if d == 'mel' else 'benign')

    train_idx, val_idx = [], []
    for _, grp in meta.groupby('dx'):
        idx = grp.index.tolist()
        n_train = int(round(len(idx) * (1.0 - test_size)))
        rng = random.Random(seed)
        rng.shuffle(idx)
        train_idx.extend(idx[:n_train])
        val_idx.extend(idx[n_train:])

    cols = ['image_id', 'label']
    train_df = meta.loc[train_idx, cols].assign(
        fitzpatrick_type=FITZ_PLACEHOLDER_HAM
    ).reset_index(drop=True)
    val_df = meta.loc[val_idx, cols].assign(
        fitzpatrick_type=FITZ_PLACEHOLDER_HAM
    ).reset_index(drop=True)

    train_df.to_csv(TRAIN_META, index=False)
    val_df.to_csv(VAL_META, index=False)
    print('train:')
    print(train_df['label'].value_counts())
    print('val:')
    print(val_df['label'].value_counts())
    print('wrote', TRAIN_META.name, 'and', VAL_META.name)


make_split()

## 2) Input transforms

`DermDataset` returns PIL images, but albumentations expects numpy arrays. `PILToAug` bridges the two. The equity-aware training pipeline prepends `MelaninNormalization` (LAB equalize to reduce bias toward light skin tones), then reuses the rest of `dark_skin_aug` with its trailing `ToTensorV2` stripped, then applies ImageNet `Normalize` + `ToTensorV2`.

The validation pipeline is a plain Resize + Normalize. Training uses a `WeightedRandomSampler` so benign/melanoma are balanced per batch.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2


class PILToAug:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, pil_img):
        out = self.transform(image=np.array(pil_img.convert('RGB')))
        return out['image']


def build_transform(train):
    base = [t for t in dark_skin_aug.transforms if not isinstance(t, ToTensorV2)]
    if train:
        steps = [MelaninNormalization()] + base + [
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    else:
        steps = [
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    return PILToAug(A.Compose(steps))


def make_loader(csv_path, img_dir, batch_size, train):
    ds = DermDataset(str(csv_path), str(img_dir), transform=build_transform(train))
    sampler = None
    if train:
        counts = ds.df['label'].value_counts()
        weights = 1.0 / ds.df['label'].map(counts).astype(np.float64)
        sampler = torch.utils.data.WeightedRandomSampler(
            torch.from_numpy(weights.to_numpy()).double(),
            len(ds),
            replacement=True,
        )
    return torch.utils.data.DataLoader(
        ds, batch_size=batch_size, shuffle=(not train), num_workers=0, sampler=sampler
    )


print('transform pipeline ready')

## 3) CBM model

`DermatologyCBM` stacks a frozen `hrnet_w18` backbone with concept heads that expose a 7-dimensional concept embedding (via `get_concepts`) plus a small classifier head. Only the head parameters receive gradients, which keeps this trainable on CPU. This cell also reports how many params are trainable vs frozen.

In [ ]:
def load_cbm(freeze_backbone=True, num_classes=2):
    from model import DermatologyCBM
    cbm = DermatologyCBM(
        num_classes=num_classes,
        num_concepts=NUM_CONCEPTS,
        hrnet_name='hrnet_w18',
        pretrained_backbone=True,
        freeze_backbone=freeze_backbone,
    )
    return cbm.to(DEVICE)


def evaluate_cbm(cbm, loader):
    cbm.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y, _ in loader:
            out = cbm(x)[0]
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
    return correct / max(total, 1)


cbm = load_cbm(freeze_backbone=True)
trainable = sum(p.numel() for p in cbm.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in cbm.parameters() if not p.requires_grad)
print(f'trainable params: {trainable:,}  |  frozen params: {frozen:,}')

## 4) Train the concept bottleneck

Plain supervised pass: CrossEntropyLoss on the binary labels over `EPOCHS` epochs. The best validation-accuracy checkpoint is written to `model_checkpoints/cbm_binary.pt`. Afterwards we prove concept extraction works end to end by running `get_concepts` on one validation batch and printing the resulting shape.

In [ ]:
cbm = load_cbm(freeze_backbone=True)
optim = torch.optim.AdamW([p for p in cbm.parameters() if p.requires_grad], lr=1e-3)
crit = torch.nn.CrossEntropyLoss()

train_loader = make_loader(TRAIN_META, HAM_DIR, BATCH_SIZE, train=True)
val_loader = make_loader(VAL_META, HAM_DIR, BATCH_SIZE, train=False)
best = -1.0

for epoch in range(1, EPOCHS + 1):
    cbm.train()
    run_loss = 0.0
    n = 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = cbm(x)[0]
        loss = crit(logits, y)
        optim.zero_grad()
        loss.backward()
        optim.step()
        run_loss += loss.item() * len(y)
        n += len(y)
    train_acc = evaluate_cbm(cbm, train_loader)
    val_acc = evaluate_cbm(cbm, val_loader)
    print(f'epoch {epoch}/{EPOCHS} | CE {run_loss / n:.4f} | '
          f'train_acc {train_acc:.4f} | val_acc {val_acc:.4f}')
    if val_acc > best:
        best = val_acc
        torch.save(cbm.state_dict(), CBM_CKPT)
        print(f'saved best cbm -> {CBM_CKPT.name} ({val_acc:.4f})')

print(f'DONE best val_acc {best:.4f}')

cbm.load_state_dict(torch.load(CBM_CKPT, map_location=DEVICE))
cbm.eval()
xb, _, _ = next(iter(val_loader))
with torch.no_grad():
    concepts = cbm.get_concepts(xb)
    print('get_concepts ->', tuple(concepts.shape), 'on', DEVICE.type)

## 5) Embed cases and train the PPO triage policy

Every synthetic case (which carries a real Fitzpatrick type 1-6) is pushed through the trained CBM to get its concept embedding, then wrapped in a `Case` (using the one-step `Rollout` API confirmed in `rl_module`). The `DermTriageEnv` + `PPOAgent` train a 3-way triage policy, and the final agent is saved to `model_checkpoints/ppo_triage.pt`.

For a quick pass lower `EPISODES` (e.g. 64) and `UPDATE_INTERVAL` (e.g. 16).

In [ ]:
def build_cases(cbm, csv_path, img_dir, limit=None):
    cbm.eval()
    meta = pd.read_csv(csv_path)
    if limit:
        meta = meta.sample(n=min(limit, len(meta)), random_state=TORCH_SEED)
    ds = DermDataset(str(csv_path), str(img_dir), transform=build_transform(train=False))
    loader = torch.utils.data.DataLoader(ds, batch_size=32, num_workers=0)
    cases = []
    with torch.no_grad():
        for x, y, fitz in loader:
            emb = cbm.get_concepts(x).cpu().numpy().astype(np.float32)
            for e, lab, f in zip(emb, y.numpy(), fitz.numpy()):
                cases.append(
                    Case(concept_embedding=e, fitz_type=int(f),
                         true_label=int(lab), patient_id=f'SYN-{len(cases)}')
                )
            if limit and len(cases) >= limit:
                break
    print('built', len(cases), 'PPO cases from', csv_path.name)
    return cases


def train_ppo_loop(cases, episodes, update_interval):
    env = DermTriageEnv(num_concepts=NUM_CONCEPTS)
    agent = PPOAgent(env.state_dim, env.action_dim)
    env.seed(cases)

    rollouts = []
    ep_rewards = []
    for episode in range(1, episodes + 1):
        state = env.reset()
        action, log_prob, value = agent.select_action(state)
        _, reward, done, _ = env.step(action)
        rollouts.append(
            Rollout(states=[state], actions=[action], rewards=[reward],
                    dones=[True], log_probs=[log_prob], values=[value])
        )
        ep_rewards.append(reward)
        if episode % update_interval == 0 and rollouts:
            metrics = agent.update(rollouts)
            pol = metrics.get('policy_loss', 0.0)
            val = metrics.get('value_loss', 0.0)
            avg_reward = float(np.mean(ep_rewards[-update_interval:]))
            print(f'    [ppo] ep {episode:5d} | avg_reward {avg_reward:6.2f} | '
                  f'policy_loss {pol:.4f} | value_loss {val:.4f}')
            rollouts = []
    return agent


cbm = load_cbm()
cbm.load_state_dict(torch.load(CBM_CKPT, map_location=DEVICE))
cases = build_cases(cbm, SYN_META, SYN_DIR, limit=200)
agent = train_ppo_loop(cases, episodes=EPISODES, update_interval=UPDATE_INTERVAL)
agent.save(str(PPO_CKPT))
print('saved agent ->', PPO_CKPT)

## 6) Fairness audit

Reload the saved agent and evaluate per-Fitzpatrick triage accuracy plus the critical-miss rate: a true lesion of class 1 (melanoma) or 2 (kaposi sarcoma) that the agent did not escalate to urgent (triage 2). The check PASSES if the critical-miss rate on Fitz IV-VI is no worse than on Fitz I-III.

In [ ]:
def fairness_report(agent, cases, title='fairness audit'):
    env = DermTriageEnv(num_concepts=NUM_CONCEPTS)
    rows = []
    for fitz in range(1, 7):
        subset = [c for c in cases if c.fitz_type == fitz]
        if not subset:
            continue
        n = len(subset)
        acc = sum(
            1 for c in subset
            if agent.evaluate(env._build_state(c)) == TRUE_LABEL_TO_TRIAGE[c.true_label]
        ) / n
        critical = sum(
            1 for c in subset
            if c.true_label in (1, 2) and agent.evaluate(env._build_state(c)) != 2
        ) / n
        rows.append((fitz, n, acc, critical))
    print('')
    print('==', title, '==')
    print('  Fitz  n     triage_acc  critical_miss')
    for fitz, n, acc, crit in rows:
        print(f'  {fitz:<4} {n:<5} {acc:<11.4f} {crit:.4f}')
    if rows:
        iv_plus = [r for r in rows if r[0] >= 4]
        i_iii = [r for r in rows if r[0] <= 3]
        cm_iv_plus = sum(r[1] * r[3] for r in iv_plus) / max(sum(r[1] for r in iv_plus), 1)
        cm_i_iii = sum(r[1] * r[3] for r in i_iii) / max(sum(r[1] for r in i_iii), 1)
        verdict = 'PASS' if cm_iv_plus <= cm_i_iii + 1e-9 else 'FAIL'
        print(f'  critical-miss  Fitz IV-VI {cm_iv_plus:.4f}  vs  '
              f'Fitz I-III {cm_i_iii:.4f}  -> {verdict}')


env = DermTriageEnv(num_concepts=NUM_CONCEPTS)
agent = PPOAgent(env.state_dim, env.action_dim)
agent.load(str(PPO_CKPT))
print('loaded agent from', PPO_CKPT)
fairness_report(agent, cases, title='fairness audit')

## Next steps

- Run `01_train_cbm.py --phase smoke` (from the repo root) as a headless sanity check of the exact same pipeline.
- Unfreeze the backbone (`freeze_backbone=False`) and/or raise `EPOCHS` once GPU access is available.
- Tune the PPO reward inside `rl_module.py` (entropy coefficient, critical-miss penalty) to close the Fitz IV-VI gap reported by the audit.